In [ ]:

import torch
import torch.nn as nn
import torchvision.models as models

# Choose a pre-trained model (ResNet18 for simplicity)
model = models.resnet18(pretrained=True)

# Check the model architecture
print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:

# Freeze all parameters in the feature extractor
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer with a new classifier head
num_features = model.fc.in_features   # ResNet18 final layer input size
model.fc = nn.Linear(num_features, 10)  # Example: 10 classes

print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained ResNet18
model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 has 10 classes
model = model.to(device)

# Data transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load CIFAR-10 but only use a subset (e.g., first 2000 samples)
train_dataset = datasets.CIFAR10(root='./data', train=True,
                                 download=True, transform=transform)
subset_indices = list(range(2000))  # small subset for speed
train_subset = Subset(train_dataset, subset_indices)
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training loop (just 1 epoch)
for epoch in range(1):
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 50 == 0:  # print every 50 batches
            print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")



100%|██████████| 170M/170M [00:05<00:00, 30.1MB/s]


Epoch 1, Batch 0, Loss: 2.4231
Epoch 1, Batch 50, Loss: 1.5342


In [ ]:

# Unfreeze the last few layers of ResNet18
for name, param in model.named_parameters():
    if "layer4" in name:   # last residual block
        param.requires_grad = True

# Now optimizer should include both classifier head + unfrozen layers
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

# Training loop (simplified, just 1 epoch for demo)
for epoch in range(1):
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 50 == 0:
            print(f"Fine-tuning Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")

Fine-tuning Epoch 1, Batch 0, Loss: 1.4953
Fine-tuning Epoch 1, Batch 50, Loss: 0.6580


In [ ]:


# Load CIFAR-10 test set
test_dataset = datasets.CIFAR10(root='./data', train=False,
                                download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Evaluation function
def evaluate(model, loader):
    model.eval()  # set to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy


In [ ]:

# Evaluate only on a subset of test data (e.g., first 2000 samples)
subset_indices = list(range(2000))
test_subset = torch.utils.data.Subset(test_dataset, subset_indices)
test_loader_small = DataLoader(test_subset, batch_size=32, shuffle=False)

frozen_accuracy = evaluate(model, test_loader_small)
print(f"Quick test accuracy (subset): {frozen_accuracy:.2f}%")

Quick test accuracy (subset): 78.85%


In [8]:

# (If you re-train with fine-tuning, run evaluate again)
fine_tuned_accuracy = evaluate(model, test_loader)  # after Step 4 training
print(f"Accuracy after fine-tuning selected layers: {fine_tuned_accuracy:.2f}%")

Accuracy after fine-tuning selected layers: 79.67%
